In [73]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter, Language
from langchain.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

In [74]:
load_dotenv()

True

In [75]:
url = 'https://www.flipkart.com/motorola-motobook-60-full-metal-oled-i5-14th-gen-intel-core-5-series-2-210h-16-gb-512-gb-ssd-windows-11-home-14irh10r-thin-light-laptop/p/itm9a50f9400e0e0?pid=COMHAUZWVNJSFAMN&otracker=wishlist&lid=LSTCOMHAUZWVNJSFAMNCQMECQ&fm=organic&iid=eb8ce955-00a1-4fdc-bd17-e3cc38eb95b2.COMHAUZWVNJSFAMN.PRODUCTSUMMARY&ppt=hp&ppn=homepage&ssid=txin2y7f340000001758805452784'

In [76]:
loader = WebBaseLoader(url)
raw_html = loader.scrape()

In [29]:
raw_html

<!DOCTYPE html>
<html lang="en"><head><link href="https://rukminim2.flixcart.com" rel="preconnect"/><link href="//static-assets-web.flixcart.com/fk-p-linchpin-web/fk-cp-zion/css/app_modules.chunk.c48a12.css" rel="stylesheet"/><link href="//static-assets-web.flixcart.com/fk-p-linchpin-web/fk-cp-zion/css/app.chunk.066267.css" rel="stylesheet"/><meta content="text/html; charset=utf-8" http-equiv="Content-type"/><meta content="IE=Edge" http-equiv="X-UA-Compatible"/><meta content="102988293558" property="fb:page_id"/><meta content="658873552,624500995,100000233612389" property="fb:admins"/><link href="https://static-assets-web.flixcart.com/www/promos/new/20150528-140547-favicon-retina.ico" rel="shortcut icon"/><link href="/osdd.xml?v=2" rel="search" type="application/opensearchdescription+xml"/><meta content="website" property="og:type"/><meta content="Flipkart.com" name="og_site_name" property="og:site_name"/><link href="/apple-touch-icon-57x57.png" rel="apple-touch-icon" sizes="57x57"/><l

In [78]:
for tag in raw_html([
    "script", "style", "nav", "footer", "header", "aside", "meta", "link",
    "noscript",   # fallback content, usually redundant
    "iframe",     # embedded ads, videos
    "form",       # login/signup/contact forms
    "input", "button", "select", "textarea",  # form fields
    "svg", "canvas",  # icons, graphics
    "img",       # images (unless you want alt text)
    "video", "audio", "source", "track",  # media elements
    "advertisement", "ads",  # ad containers (if present as tags)
]):
    tag.decompose()


raw_html = str(raw_html)
raw_html

'<!DOCTYPE html>\n<html lang="en"><head></head><body><div id="container"><div><div class="krHvwW"><div class="J+HqMZ"><div class="LOe-Xr"></div><div class="Ja1j8k"><div class="ngOQ7L"><div class="F9+fd2"><a href="/"></a><a class="MwbhDR" href="/plus">Explore<!-- --> <span class="s4NJ5L">Plus</span></a></div></div><div class="kRd8Cs"></div><div class="RbF1Du UB4mMK"><div class="UL9nZx"><div class="tP+nZg _2E9UgX"><div><a class="wsejfv" href="/account/login?ret=/motorola-motobook-60-full-metal-oled-i5-14th-gen-intel-core-5-series-2-210h-16-gb-512-gb-ssd-windows-11-home-14irh10r-thin-light-laptop/p/itm9a50f9400e0e0%3Fpid%3DCOMHAUZWVNJSFAMN%26otracker%3Dwishlist%26lid%3DLSTCOMHAUZWVNJSFAMNCQMECQ%26fm%3Dorganic%26iid%3Deb8ce955-00a1-4fdc-bd17-e3cc38eb95b2.COMHAUZWVNJSFAMN.PRODUCTSUMMARY%26ppt%3Dhp%26ppn%3Dhomepage%26ssid%3Dtxin2y7f340000001758805452784">Login</a></div></div></div></div><div class="RbF1Du"><a class="CDJO0-" href="https://seller.flipkart.com/sell-online/?utm_source=fkwebsite&

In [81]:
soup = BeautifulSoup(raw_html, "html.parser")

# remove all attributes from all tags
for tag in soup.find_all(True):  # True = all tags
    tag.attrs = {}

raw_html = str(soup)
raw_html

"<!DOCTYPE html>\n\n<html><head></head><body><div><div><div><div><div></div><div><div><div><a></a><a>Explore<!-- --> <span>Plus</span></a></div></div><div></div><div><div><div><div><a>Login</a></div></div></div></div><div><a><span>Become a Seller</span></a></div><div><div><div><div><div> <!-- -->More<!-- --> </div></div></div></div></div><div><div><div><a><span>Cart</span></a></div></div></div></div><div></div></div><div></div></div><div></div><div><div><div><div><div><div><div><div><div><ul><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li></ul></div><div></div><div></div></div></div><div><div><div></div><div></div></div></div></div><div><div></div></div></div></div><div><div><ul><li></li><li></li><

In [82]:
# Split each section by HTML-aware splitter
html_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.HTML,
    chunk_size=5000,
    chunk_overlap=500
)

html_splitter._separators = html_splitter._separators[:-1] + ["\n\n\n", "\n\n", "\n", " ", ""]

html_chunks = html_splitter.split_text(raw_html)

print(f"Number of HTML-aware chunks: {len(html_chunks)}")

Number of HTML-aware chunks: 7


In [54]:
html_splitter._separators[:-1] + ["\n\n\n", "\n\n", "\n", " ", ""]

['<body',
 '<div',
 '<p',
 '<br',
 '<li',
 '<h1',
 '<h2',
 '<h3',
 '<h4',
 '<h5',
 '<h6',
 '<span',
 '<table',
 '<tr',
 '<td',
 '<th',
 '<ul',
 '<ol',
 '<header',
 '<footer',
 '<nav',
 '<head',
 '<style',
 '<script',
 '<meta',
 '<title',
 '\n\n\n',
 '\n\n',
 '\n',
 ' ',
 '']

In [84]:
html_chunks[1]

'<body><div><div><div><div><div></div><div><div><div><a></a><a>Explore<!-- --> <span>Plus</span></a></div></div><div></div><div><div><div><div><a>Login</a></div></div></div></div><div><a><span>Become a Seller</span></a></div><div><div><div><div><div> <!-- -->More<!-- --> </div></div></div></div></div><div><div><div><a><span>Cart</span></a></div></div></div></div><div></div></div><div></div></div><div></div><div><div><div><div><div><div><div><div><div><ul><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li><li><div><div></div></div></li></ul></div><div></div><div></div></div></div><div><div><div></div><div></div></div></div></div><div><div></div></div></div></div><div><div><ul><li></li><li></li></ul></div></div><div></div></div><div>

In [57]:
type(html_chunks[0])

str

In [91]:
clean_chunks = []
for text in html_chunks:
    soup = BeautifulSoup(text, "html.parser")
    clean_text = soup.get_text(separator="<<>>", strip=True)
    if len(clean_text) > 5:
        clean_chunks.append(clean_text)

lengths = []
for text in clean_chunks:
    lengths.append(len(text))

print(sorted(lengths))

# print(html_chunks[45])
# print('\n\n')
print(clean_chunks[-2])


[1823, 1939, 1953, 2682, 2821, 2850]
Jawad Ali Syed<<>>Certified Buyer<<>>, Ballepalle<<>>3 months ago<<>>11<<>>1<<>>Permalink<<>>Report Abuse<<>>5<<>>Perfect product!<<>>The  Motobook 60 delivers excellent performance with a sleek design, fast processing, and vibrant crystal clear display. Ideal for work and entertainment, it offers long battery life and smooth multitasking. A reliable, high-value choice for everyday users. Great balance of power and portability.<<>>Only negative point is that the speakers are not very loud..<<>>READ MORE<<>>Maxson  Koli<<>>Certified Buyer<<>>, Mumbai<<>>4 months ago<<>>18<<>>4<<>>Permalink<<>>Report Abuse<<>>5<<>>Excellent<<>>Good choice under 55k.<<>>Pros:<<>>Oled Screen is awesome. It supports hdr. Fantastic for entertainment purposes.<<>>Performance is good, no issues there.<<>>Battery backup is satisfactory. If you want more backup, try energy saving mode and reduce display resolution to 1080p.(6+hrs)<<>>Ram and storage are expandable.<<>>Cons:<<

In [97]:
final_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["<<>>", "\n\n\n", "\n\n", "\n", ".", " ", ""]
)
final_chunks = []
for text in clean_chunks:
    new_chunks = final_splitter.split_text(text)
    final_chunks.extend(new_chunks)

len(final_chunks)

chunks = []
for text in final_chunks:
    new_text = text.replace("<<>>", " ")
    chunks.append(new_text)

for text in chunks:
    print(text)

Explore Plus Login Become a Seller More Cart Home Computers Laptops MOTOROLA Laptops MOTOROLA Motobook 60 Full Metal OLED (i5 14th Gen) Intel Core 5 (Series 2) 210H - (16 GB/512 GB SSD/Windows 11 Home) 14IRH10R Thin and Light Laptop (14 Inch, PANTONE Wedgewood, 1.4 Kg, With MS Office) Compare Share MOTOROLA Motobook 60 Full Metal OLED (i5 14th Gen) Intel Core 5 (Series 2) 210H - (16 GB/512 GB SSD/Windows 11 Home) 14IRH10R Thin and Light Laptop (14 Inch, PANTONE Wedgewood, 1.4 Kg, With MS Office) 4.4 715 Ratings & 76 Reviews Special price ₹49,989 ₹ 93,690 46% off + ₹99 Protect Promise Fee Learn more Secure delivery by 3 Oct, Friday Available offers Bank Offer 10% Off on Supermoney UPI. Max discount of ₹50. Minimum order value of ₹250. T&C Bank Offer 5% cashback on Flipkart SBI Credit Card upto ₹4,000 per calendar quarter T&C Bank Offer 5% cashback on Axis Bank Flipkart Debit Card
 T&C Bank Offer 5% cashback on Flipkart SBI Credit Card upto ₹4,000 per calendar quarter T&C Bank Offer 5% c

In [103]:
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_texts(chunks, embeddings)

In [104]:
vectorstore.index_to_docstore_id

{0: 'bd8642be-28fb-4503-abcc-bd0566375c59',
 1: 'd9d7f9f9-9507-4616-91e0-30a21fab2b38',
 2: '9df7ae5d-b804-45f6-83b0-19435adf9b33',
 3: 'e763ffe8-efbe-44d2-bb4e-9b0e9e74f109',
 4: '9804da3b-65b7-420c-a0f2-001603c58a23',
 5: 'a0b43ab5-2420-4303-868d-6f8728870641',
 6: 'fa2b094f-550e-47ba-bf47-52cd233b2c7e',
 7: '8c983c2e-211c-4e8a-bad9-110ddee5c743',
 8: '99435416-253d-4347-b350-6fea7adff571',
 9: 'f8d6c317-ad05-4f69-bf00-5846398f39fe',
 10: '1acc3a86-9d06-4fc0-8395-59a00a3580e8',
 11: '20f71491-1151-4058-a8f9-8d569fbd6b70',
 12: 'a7f6daf2-0c66-4545-9533-14f7bd38802b',
 13: 'c6cad999-af6b-46dd-8db5-eacfa87be2c0',
 14: 'ce0c6a8d-e7cc-4356-b947-9f5ee0dbc866',
 15: 'bda0627d-9f3f-4970-bd55-019ae4f672fb',
 16: 'c6744636-6f6a-464f-a736-a53977ac9048',
 17: 'e8d2386e-b882-4a21-9cb7-2cbb78aa374f',
 18: 'e78076a3-d993-45f3-b278-c48ee754bbfb',
 19: '831f0a2b-d776-4721-877f-15b90e492a4d',
 20: 'f5e0ccfd-8dd8-483f-906c-9bd937b2f8cf'}

In [105]:
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k':5})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002705E82DF70>, search_kwargs={'k': 5})

In [106]:
retriever.invoke('what is the price of the laptop?')

[Document(id='c6cad999-af6b-46dd-8db5-eacfa87be2c0', metadata={}, page_content=" READ MORE Karthik  Bhat Certified Buyer , Udupi 4 months ago 31 7 Permalink Report Abuse 5 Worth every penny Laptop comes with the latest spec in means of processor released in 2025, ddr5 RAM, can expand both RAM and ROM, display is next level, light weight, Microsoft office 2024 life time free, seamless access to Motorola phone, backlight keyboard, only flaw is speaker sound is low but comes with Dolby atomos if you use headphones sound is next level like theatre. bigt track pad, premium feel, i got in 53k, with emi discount final price was 47k only, it's a deal, with this price go for it without s... READ MORE Jawad Ali Syed Certified Buyer , Ballepalle 3 months ago 11 1"),
 Document(id='9df7ae5d-b804-45f6-83b0-19435adf9b33', metadata={}, page_content=' Light Laptop without Optical Disk Drive Easy Payment Options No cost EMI starting from ₹8,332/month Cash on Delivery Net banking & Credit/ Debit/ ATM car

In [137]:
from utils.prompts import std_prompt

prompt = PromptTemplate(
    input_variables=["context", "query"],
    template=std_prompt
)

std_prompt

"\nYou are a helpful assistant.\n\nRelevant Webpage Content context:\n{context}\n\nUser Query: {query}\n\nAnswer clearly using webpage content context.\nIf the user asks for a summary/overview, summarize the whole webpage content.\nIf the context doesn't contain the relevant information to answer the user query, then say 'Webpage doesn't contain the relevant information.'\n"

In [138]:
llm = HuggingFaceEndpoint(
    model="meta-llama/Llama-3.3-70B-Instruct",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

In [139]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [140]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [141]:
parser = StrOutputParser()

In [142]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'query': RunnablePassthrough()
})

In [143]:
main_chain = parallel_chain | prompt | model | parser

In [144]:
main_chain.invoke('What is the price of this product?')

'The price of the Motorola Motobook 60 laptop is ₹49,989, which is a discounted price from the original price of ₹93,690, offering a 46% discount. Additionally, one of the reviewers, Karthik Bhat, mentioned that he got the laptop for ₹47,000 after an EMI discount, with an initial price of ₹53,000, and another payment option is available with no-cost EMI starting from ₹8,332/month.'

In [145]:
main_chain.invoke('what is the price of the macbook air m4 ?')

"Webpage doesn't contain the relevant information. \n\nThe webpage content is about the Motorola Motobook 60 laptop, its features, and reviews, but it does not mention the price of the MacBook Air M4."